In [ ]:

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader, SubsetRandomSampler
from torchvision.datasets import ImageFolder
from torchvision import transforms
from skopt import gp_minimize, load
from skopt.space import Real
from skopt.callbacks import CheckpointSaver
from sklearn.model_selection import KFold
import json
import time
import re
import wandb

wandb.login()

from tqdm import tqdm


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\veren\_netrc.
wandb: Currently logged in as: verena_vanessa (verena_vanessa-danmarks-tekniske-universitet-dtu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
# ==========================================
# Checkpoint Configuration Variables
# ==========================================
CHECKPOINT_BASE_NAME = '3d_cv_optimization'
USE_CHECKPOINT = False   # Set to True to resume from a checkpoint, False to start new
DESIRED_CHECKPOINT_ID = None  # Set to None for latest, or an integer for a specific checkpoint ID
USE_NARROWED_SPACE = True   # Set to True to use narrowed_search_space from the analysis cell (5b)

# Local checkpoint directory
DRIVE_DIR = r"BO_Checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)


### Data Preprocessing & Loading (normalisering + transforms)

**Hvad er idéen?**
Vi skal først loade billeddata og definere en preprocessing-pipeline, så vores model får input i en stabil skala. Det gør træning hurtigere og mere stabilt.

Koncepter der bruges

ImageFolder: Læser et billed-dataset, hvor hver klasse ligger i sin egen mappe.

Transforms:

- Resize(128,128): Ens størrelse på alle billeder.

- ToTensor(): Konverterer billeder til PyTorch-tensors med værdier i intervallet [0,1].

- Normalize(mean, std): Standardiserer hver farvekanal (R, G, B), så data typisk får middelværdi ~0 og std ~1.

- Caching: Vi gemmer beregnet mean/std i en JSON-fil, så vi ikke regner det ud igen ved næste kørsel.

**Hvad gør vi konkret i koden?**

1. Vi tjekker dataset-mappen for at sikre, at vi kan finde Training/Testing.

2. Vi forsøger at loade allerede beregnede normaliseringsstatistikker fra en JSON-cache.

3. Hvis de ikke findes, gennemløber vi dataset i batches og beregner mean/std pr. farvekanal baseret på alle pixels.

4. Vi bygger en endelig transform, som inkluderer normalisering, og loader training_dataset og testing_dataset med den.

5. Vi samler dem også i et samlet ConcatDataset (praktisk hvis vi senere vil splitte eller lave cross-validation).

**Hvorfor er det relevant for Bayesian Optimization vs Grid Search?**
Når vi optimerer hyperparametre, skal alle modeller og alle trials trænes på data, der er preprocesset på præcis samme måde. Ellers kan forskelle i performance skyldes preprocessing i stedet for selve optimeringsmetoden.

**Hvad kunne vi ellers have gjort (og hvad ville effekten være)?**

Beregne mean/std kun på træningsdata (ikke test).
Effekt: mere “ren” evaluering uden leakage — typisk en mere troværdig rapport, selv hvis accuracy falder en smule.

Bruge ImageNet mean/std i stedet for dataset-specifikke stats.
Effekt: hurtigere opsætning og ofte fint hvis man bruger pre-trained modeller; men kan være lidt dårligere, hvis jeres dataset har meget anderledes farvefordeling.

In [ ]:
# ==========================================
# 1. Data Preprocessing & Loading
# ==========================================

# Lokal sti til dataset-mappen.
# Forventet struktur:
# dataset/
#   Training/<class folders...>
#   Testing/<class folders...>
dataset_path = r"dataset"
print(f"Contents of {dataset_path}: {os.listdir(dataset_path)}")

# Fil til at cache (gemme) dataset-specifik mean/std, så vi ikke skal regne dem igen hver gang.
# OBS: DRIVE_DIR skal være defineret tidligere (fx i Colab eller som projekt-rod).
NORM_STATS_FILE = os.path.join(DRIVE_DIR, "dataset_norm_stats.json")

# ------------------------------------------------------------
# A) Load cached normalization statistics OR compute them
# ------------------------------------------------------------
if os.path.exists(NORM_STATS_FILE):
    # Hvis stats allerede er beregnet, genbruger vi dem for hurtigere opstart.
    with open(NORM_STATS_FILE, "r") as f:
        _stats = json.load(f)

    DATASET_MEAN = _stats["mean"]  # liste med 3 tal (RGB)
    DATASET_STD  = _stats["std"]   # liste med 3 tal (RGB)
    print(f"Loaded cached normalization stats from {NORM_STATS_FILE}")

else:
    print("Computing dataset-specific normalization statistics (first run)...")

    # Midlertidig transform: vi vil kun resize + ToTensor, så vi kan måle rå pixel distribution.
    # (Vi kan ikke Normalize før vi kender mean/std.)
    _tmp_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),  # konverterer PIL -> Tensor med værdier i [0,1]
    ])

    # Vi loader både Training og Testing for at beregne samlet mean/std.
    # OBS: Dette kan give mild data leakage ift. evalueringsfairness.
    _tmp_train = ImageFolder(os.path.join(dataset_path, "Training"), transform=_tmp_transform)
    _tmp_test  = ImageFolder(os.path.join(dataset_path, "Testing"),  transform=_tmp_transform)

    # Saml alle billeder i én dataset-objekt
    _tmp_all = ConcatDataset([_tmp_train, _tmp_test])

    # DataLoader til batch-vis gennemløb (hurtigere end én og én)
    _tmp_loader = DataLoader(_tmp_all, batch_size=256, shuffle=False, num_workers=0)

    # Akkumulatorer til at beregne mean/std på tværs af alle pixels i alle billeder.
    # _mean og _std holder summer pr. kanal (RGB).
    _mean = torch.zeros(3)
    _std  = torch.zeros(3)
    _n_pixels = 0  # total antal pixels pr. kanal (B*H*W summeret)

    # Loop gennem batches og akkumulér.
    # imgs har shape: (batch, channels=3, height, width)
    for imgs, _ in tqdm(_tmp_loader, desc="Norm stats", leave=False):
        b, c, h, w = imgs.shape
        _n_pixels += b * h * w

        # Sum af pixelværdier pr. kanal
        _mean += imgs.sum(dim=[0, 2, 3])

        # Sum af kvadrerede pixelværdier pr. kanal
        _std  += (imgs ** 2).sum(dim=[0, 2, 3])

    # E[x] pr kanal
    DATASET_MEAN = (_mean / _n_pixels).tolist()

    # std = sqrt(E[x^2] - (E[x])^2)
    DATASET_STD = (
        (_std / _n_pixels - torch.tensor(DATASET_MEAN) ** 2).sqrt()
    ).tolist()

    # Ryd op i midlertidige variabler (især rart i notebook for RAM)
    del _tmp_transform, _tmp_train, _tmp_test, _tmp_all, _tmp_loader, _mean, _std, _n_pixels

    # Gem stats til senere runs
    with open(NORM_STATS_FILE, "w") as f:
        json.dump({"mean": DATASET_MEAN, "std": DATASET_STD}, f, indent=2)

    print(f"Saved normalization stats to {NORM_STATS_FILE}")

print(f"Dataset mean: {DATASET_MEAN}")
print(f"Dataset std:  {DATASET_STD}")

# ------------------------------------------------------------
# B) Final transform (inkl. Normalize) til modeltræning og test
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD)  # standardisering pr. kanal
])

# Trænings- og test-datasets med samme transform
training_dataset = ImageFolder(os.path.join(dataset_path, "Training"), transform=transform)
testing_dataset  = ImageFolder(os.path.join(dataset_path, "Testing"),  transform=transform)

# Samlet dataset (bruges måske senere til et fælles split / CV)
dataset = ConcatDataset([training_dataset, testing_dataset])

print(f"Total dataset size: {len(dataset)} images")

Contents of dataset: ['Testing', 'Training']
Loaded cached normalization stats from BO_Checkpoints\dataset_norm_stats.json
Dataset mean: [0.18654859066009521, 0.18655261397361755, 0.18659797310829163]
Dataset std:  [0.19559581577777863, 0.19559478759765625, 0.1956312358379364]
Total dataset size: 7200 images


### Model Definition – SimpleTumorCNN

**Hvad er idéen?**
Vi definerer en lille CNN, der kan lære mønstre i billeder (fx tumor-typer) ved at kombinere convolution-lag, non-linearities og pooling, og til sidst outputte en klasse.

Koncepter der bruges

- Convolution (Conv2d): lærer små “filtre” som kan opdage kanter, teksturer og mønstre i billeder.

- BatchNorm: stabiliserer og accelererer træning ved at normalisere aktiveringer pr. kanal (men kan være sensitiv for små batch sizes).

- ReLU: gør modellen i stand til at lære ikke-lineære mønstre.

- MaxPool: reducerer billedets opløsning og gør features mere robuste (samt hurtigere).

- Adaptive Average Pooling: opsummerer hver feature map til én værdi → giver fast feature-vector størrelse uanset inputstørrelse.

- Dropout: tilfældigt “slukker” nogle features under træning → reducerer overfitting.

- CrossEntropyLoss: standard loss til multi-class klassifikation (tager logits direkte).

**Hvad gør vi konkret i koden?**

1. Vi bygger en CNN med 3 blocks, hvor vi øger kanalerne 16→32→64 og reducerer billedstørrelsen med pooling.

2. Vi bruger global average pooling for at få en fast 64-dimensionel feature-vektor.

3. Vi bruger en simpel klassifikations-head (Dropout + Linear) til at producere logits for 4 klasser.

4. Vi udskriver parametertallet som sanity check (modellen er meget lille, ~24k params).

5. Vi definerer loss-funktionen CrossEntropyLoss, som matcher outputformatet.

**Hvorfor er det relevant for Bayesian Optimization vs Grid Search?**
En lille og stabil model gør det muligt at køre mange hyperparameter-trials indenfor samme compute-budget. Det er vigtigt for en fair sammenligning af søgestrategier, fordi BO typisk “betaler” sig når man har et begrænset antal dyre evalueringer.

**Hvad kunne vi ellers have gjort (og hvad ville effekten være)?**

1. Skifte BatchNorm ud med GroupNorm/LayerNorm (især hvis batch size bliver lille).
Effekt: mere stabil træning ved små batches → mindre støj i jeres hyperparameter-evalueringer.

2. Tilføje lidt mere kapacitet eller regularisering (fx et ekstra conv-block, eller weight decay / dropout også i features).
*Effekt*: potentielt højere accuracy, men langsommere trials og større risiko for overfitting hvis dataset er lille.

In [ ]:
# ==========================================
# 2. Model Definition — SimpleTumorCNN
# ==========================================
class SimpleTumorCNN(nn.Module):
    """
    Lightweight custom CNN (~24k parameters).

    Arkitektur:
      - 3 convolution blocks: Conv -> BatchNorm -> ReLU -> MaxPool
      - Global pooling: AdaptiveAvgPool2d((1,1))
      - Klassifikations-head: Flatten -> Dropout -> Linear

    Motivation:
      - Lille model = hurtig træning pr. trial
      - God til hyperparameter-søgning (Bayesian opt vs. grid) fordi der kan køres mange eksperimenter
    """
    def __init__(self, num_classes: int = 4, dropout_rate: float = 0.1):
        super().__init__()

        # Feature extractor: gradvist flere kanaler + nedskalering via pooling
        self.features = nn.Sequential(
            # Block 1: input RGB (3 kanaler) -> 16 feature maps
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),          # normaliserer aktiveringer pr. kanal (kan være batch-size sensitiv)
            nn.ReLU(inplace=True),       # non-linearity
            nn.MaxPool2d(kernel_size=2), # halverer H og W

            # Block 2: 16 -> 32 kanaler
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 32 -> 64 kanaler
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Global average pooling: reducerer (H, W) til (1, 1)
            # Fordel: gør head'en uafhængig af input resolution (indenfor rimelighed).
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Klassifikations-head: fra 64 pooled features -> num_classes logits
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # (B, 64, 1, 1) -> (B, 64)
            nn.Dropout(dropout_rate),     # regulering for at reducere overfitting
            nn.Linear(64, num_classes),   # output = logits (ingen softmax her)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Kør feature extractor og derefter klassifikationshead
        x = self.features(x)
        x = self.classifier(x)
        return x


# ------------------------------------------------------------
# Sanity check: parameter count
# ------------------------------------------------------------
_tmp_model = SimpleTumorCNN(num_classes=4, dropout_rate=0.1)

# Antal parametre (inkl. bias) i hele modellen
_param_count = sum(p.numel() for p in _tmp_model.parameters())
print(f"SimpleTumorCNN parameter count: {_param_count:,}")

del _tmp_model  # ryd op i notebook

# Loss funktion til multi-class klassifikation.
# CrossEntropyLoss forventer:
# - model output: rå logits med shape (batch, num_classes)
# - labels: class indices med shape (batch,)
criterion = nn.CrossEntropyLoss()

SimpleTumorCNN parameter count: 24,068


### Training Params & BO Configuration

**Hvad er idéen?**
Vi fastlægger træningsbudget og hyperparameter-søgerummet. Det er her, vi sikrer at Bayesian Optimization og grid search sammenlignes fair, fordi vi bestemmer hvor mange “forsøg” (trials) og hvor dyr hver evaluering er (epochs og cross-validation).

Koncepter der bruges

- Trial budget (CALLS): Hvor mange gange vi træner en model med et nyt sæt hyperparametre.

- Epochs (EPOCHS): Hvor mange gennemløb af træningsdata vi giver hver trial (træningsbudget pr. trial).

- Cross-validation (N_FOLDS): Vi evaluerer hvert hyperparameter-sæt på flere folds for at få et mere robust estimate.

- Search space (Real + priors):

    - log-uniform til learning rate og weight decay, fordi de typisk varierer over flere størrelsesordener.

    - uniform til dropout, fordi et lineært interval ofte er fint.

- Checkpoint/run IDs: Vi genererer et ID baseret på eksisterende filer, så vi kan gemme eller genoptage runs uden at overskrive.

**Hvad gør vi konkret i koden?**

1. Vi fastsætter budgettet: 18 BO trials, 50 epochs per trial, batch size 32 og 3-fold CV.

2. Vi definerer et 3D hyperparameter-rum (LR, weight decay, dropout).

3. Vi opretter globale variabler til at holde styr på trial-nummer og grouping i fx WandB.

4. Vi definerer en funktion der kan finde et nyt run-ID eller den seneste eksisterende checkpoint-ID.

**Hvorfor er det relevant for BO vs grid search?**
Det er her vi styrer fairness: hvis BO får flere evalueringer (fx warm-start oveni), eller hvis epochs/CV håndteres forskelligt, så kan performance-forskelle skyldes budgetforskelle fremfor selve søgealgoritmen.

**Hvad kunne vi ellers have gjort (og hvad ville effekten være)?**

1. Definere budget i “antal modeltræninger” eller “tid” i stedet for antal trials (fx trials × folds, eller wall-clock).
*Effekt*: mere fair og mere troværdig sammenligning, især hvis BO har warm-start eller hvis nogle trials stopper tidligt.

2. Brug early stopping + fast patience (samme regler for grid og BO).
*E*ffekt*: hurtigere eksperimenter og mindre overfitting; men kan ændre støjniveauet i objective, hvilket påvirker BO’s effektivitet.

In [ ]:
# ==========================================
# 3. Training Params & BO Configuration
# ==========================================

# Antal nye BO-evalueringer (trials). Målet er at matche grid search-budgettet.
# OBS: Hvis man bruger warm-start (initial_points) ud over dette, bør det tælles med
# for en fair sammenligning, med mindre det er "genbrug" af allerede kørte trials.
CALLS = 18

# Træningsbudget pr. trial (og pr. fold, hvis I bruger CV)
EPOCHS = 50

# DataLoader settings
BATCH_SIZE = 32
NUM_WORKERS = 3  # kan give hurtigere loading; afhænger af OS/Jupyter stabilitet

# Cross-validation settings
N_FOLDS = 3

# Seed til reproducérbarhed (skal faktisk "sættes" i random/numpy/torch senere)
SEED = 42

# ------------------------------------------------------------
# Search space (hyperparametre) til Bayesian Optimization / Grid
# ------------------------------------------------------------
# 3D search space:
# - learning_rate: log-uniform fordi LR typisk varierer over størrelsesordener
# - weight_decay: log-uniform af samme grund
# - dropout: uniform i et rimeligt interval
search_space = [
    Real(1e-4, 1e-1, prior='log-uniform', name='learning_rate'),
    Real(1e-5, 1e-2, prior='log-uniform', name='weight_decay'),
    Real(0.0,  0.5,  prior='uniform',     name='dropout'),
]

# ------------------------------------------------------------
# Global state til logging/trial tracking (fx WandB)
# ------------------------------------------------------------
current_call = 0

# Bruges som et run-ID / gruppe-ID (fx til WandB grouping).
# Sættes typisk i "main"-blokken, så alle trials for dette run kan grupperes sammen.
checkpoint_id_for_this_run = 0


def get_checkpoint_id(base_name: str, find_latest: bool = False):
    """
    Find/Generate checkpoint IDs baseret på eksisterende filer i DRIVE_DIR.

    Filer forventes at have format:
        {base_name}_{id}.pkl

    Parametre:
      - base_name: prefix for filerne (fx "bo_results" eller "grid_results")
      - find_latest: hvis True -> returnér højeste eksisterende ID (til resume)
                     hvis False -> returnér første ledige ID (til ny run)

    Returnerer:
      - int ID, eller None hvis find_latest=True og ingen filer findes.
    """
    existing_ids = []

    # Scan DRIVE_DIR og find filer der matcher base_name_<id>.pkl
    for f_name in os.listdir(DRIVE_DIR):
        match = re.match(rf'^{re.escape(base_name)}_(\d+)\.pkl$', f_name)
        if match:
            existing_ids.append(int(match.group(1)))

    if find_latest:
        # Resume: tag den største ID vi kan finde
        return max(existing_ids) if existing_ids else None

    # Nyt run: find "første hul" i 0,1,2,... så IDs forbliver kompakte
    if not existing_ids:
        return 0

    existing_ids.sort()
    for i, _id in enumerate(existing_ids):
        if i != _id:
            return i

    # Hvis der ikke var huller, så brug næste ledige efter sidste
    return len(existing_ids)

### Objective Function – 3-Fold CV i Bayesian Optimization

**Hvad er idéen?**
Bayesian Optimization skal bruge en “objective function”, der kan give en score for et sæt hyperparametre. I vores setup er scoren gennemsnittet af validation loss fra 3-fold cross-validation.

Koncepter

- Objective function: En funktion der tager hyperparametre og returnerer et tal (lavere er bedre).

- K-fold cross-validation (K=3): Vi splitter datasættet i 3 dele. Vi træner på 2 dele og validerer på den sidste – og gentager så, så alle dele bruges som validation én gang.

- AdamW: Optimeringsalgoritme, hvor weight decay fungerer som en mere korrekt form for L2-regularisering.

- WandB logging: Vi logger metrics for at kunne sammenligne trials og lave plots bagefter.

**Hvad gør vi konkret i koden?**

1. BO giver et sæt hyperparametre: learning rate, weight decay og dropout.

2. Vi starter et nyt WandB-run for dette trial og logger hyperparametrene.

3. Vi laver 3-fold CV (med shuffle og en fast seed for at få samme split hver gang).

4. For hvert fold:

    - Vi laver nye DataLoaders via SubsetRandomSampler.

    - Vi laver en ny model fra scratch og en ny optimizer.

    - Vi træner i 50 epoker og logger træningstab pr. epoch.

    - Vi evaluerer på val-splittet og logger val loss og accuracy.

    - Vi tager gennemsnittet af val loss og accuracy over folds og returnerer mean CV val loss til BO.

**OBS (metode/fortolkning):**
I vores implementation kører KFold-splittet over dataset, som er en samling af både Training- og Testing-mappen. Det betyder at “Testing” indgår i cross-validation og derfor ikke fungerer som et rent hold-out testset i denne opsætning.

**Hvad kunne vi ellers have gjort (og hvad ville effekten være)?**

1. Køre CV kun på training-delen og holde testing helt ude til slut-evaluering.
*Effekt*: mere troværdig generaliseringsmåling (typisk lidt lavere, men mere “ærligt”).

2. Brug StratifiedKFold i stedet for KFold.
*Effekt*: mere ens klassefordeling i hvert fold → mindre varians og mere stabil objective, hvilket ofte hjælper BO.

In [ ]:
# ==========================================
# 4. Objective Function (3-Fold CV)
# ==========================================
def train_model(params):
    """
    Objective function for Bayesian Optimization.
    Trains SimpleTumorCNN with 3-Fold CV and returns mean validation loss.

    OBS (vigtigt for rapporten / metode):
    - I denne implementation laves CV over `dataset`, som tidligere blev defineret som:
        dataset = ConcatDataset([training_dataset, testing_dataset])
      Det betyder at data fra både Training og Testing indgår i CV-splittet.
      Konsekvens: "Testing" er ikke længere et rent hold-out testset, og der kan opstå
      data leakage ift. en klassisk evalueringsopsætning.
      (Det kan give optimistiske resultater ift. generalisering.)
    """
    global current_call, checkpoint_id_for_this_run
    current_call += 1

    # params kommer typisk fra skopt og er en liste/tuple i samme rækkefølge som search_space
    learning_rate = params[0]
    weight_decay  = params[1]
    dropout       = params[2]

    # Clear GPU memory from previous trial
    # OBS: empty_cache frigør cached memory, men gør ikke nødvendigvis memory usage = 0.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Initialize WandB for this trial
    run = wandb.init(
        # entity="2121jmmn-danmarks-tekniske-universitet-dtu",
        project="3d_cv_simpleTumorCNN",

        # group bruges til at samle alle trials under samme overordnede run/checkpoint
        group=f"{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}",
        name=f"trial_{current_call}",

        # reinit=True: tillader wandb.init mange gange i samme notebook process
        reinit=True,
        resume="never",
        config={
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "dropout": dropout,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "n_folds": N_FOLDS,
            "optimizer": "AdamW",
            "trial": current_call,
        }
    )

    print(f"\n{'='*60}")
    print(f"  Trial {current_call}/{CALLS}")
    print(f"  lr={learning_rate:.6f}  wd={weight_decay:.6f}  dropout={dropout:.4f}")
    print(f"{'='*60}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- 3-Fold Cross-Validation ---
    # shuffle=True + random_state=SEED giver samme fold-split på tværs af runs
    # OBS: KFold er ikke stratificeret -> klassefordeling kan variere mellem folds,
    # især hvis datasættet er ubalanceret. Det kan gøre resultater mere "noisy".
    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_losses = []
    fold_accuracies = []

    # Splitter over index-range for hele `dataset`
    # OBS: Her bruges `dataset` (ConcatDataset af training+testing) => potentielt leakage ift. klassisk setup.
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(range(len(dataset)))):
        print(f"\n  --- Fold {fold_idx + 1}/{N_FOLDS} ---")

        # Samplers for this fold
        # SubsetRandomSampler vælger batches tilfældigt fra de givne indices
        train_sampler = SubsetRandomSampler(train_idx)
        val_sampler   = SubsetRandomSampler(val_idx)

        workers = NUM_WORKERS

        # persistent_workers=True kan give bedre performance, men:
        # OBS: kræver num_workers > 0, ellers kan det give fejl hvis workers sættes til 0 i andre setups.
        train_loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                  sampler=train_sampler,
                                  num_workers=workers, persistent_workers=True)
        val_loader   = DataLoader(dataset, batch_size=BATCH_SIZE,
                                  sampler=val_sampler,
                                  num_workers=workers, persistent_workers=True)

        # Fresh model & optimizer per fold
        # (Vigtigt i CV: hver fold skal starte "fra scratch" for at være sammenlignelig)
        model = SimpleTumorCNN(num_classes=4, dropout_rate=dropout).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        # --- Training loop ---
        for epoch in range(EPOCHS):
            model.train()
            running_loss = 0.0

            # Målinger af data-loading vs compute (diagnostik)
            data_time = 0.0
            compute_time = 0.0

            pbar = tqdm(train_loader, desc=f"  Fold {fold_idx+1} Epoch {epoch+1}/{EPOCHS}", leave=False)
            end = time.time()

            for _batch_idx, (inputs, labels) in enumerate(pbar):
                # Tid brugt på at hente batch fra loader
                data_time += time.time() - end

                comp_start = time.time()
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                compute_time += time.time() - comp_start

                # Estimat af hvor stor andel af tiden der går på data-loading
                total_time = data_time + compute_time
                data_pct = 100 * data_time / total_time if total_time > 0 else 0

                # Epoch tids-estimat (tqdm)
                elapsed = pbar.format_dict.get('elapsed', 0)
                remaining = (pbar.format_dict.get('total', 1) - pbar.format_dict.get('n', 0)) \
                            * pbar.format_dict.get('elapsed', 0) \
                            / max(pbar.format_dict.get('n', 1), 1)
                epoch_total = elapsed + remaining
                et_min, et_sec = divmod(int(epoch_total), 60)

                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'epoch_est': f'{et_min:02d}:{et_sec:02d}',
                    'data%': f'{data_pct:.0f}%'
                })
                end = time.time()

            avg_train_loss = running_loss / len(train_loader)

            # WandB logging pr epoch
            # OBS: data_loading_pct her er data_pct fra sidste batch, ikke nødvendigvis epoch-gennemsnit.
            wandb.log({
                "fold": fold_idx + 1,
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "data_loading_pct": data_pct,
            })

        # --- Validation for this fold ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                # predicted class = argmax over logits
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_fold_val_loss = val_loss / len(val_loader)
        fold_accuracy = 100 * correct / total

        fold_losses.append(avg_fold_val_loss)
        fold_accuracies.append(fold_accuracy)

        wandb.log({
            "fold": fold_idx + 1,
            "fold_val_loss": avg_fold_val_loss,
            "fold_val_accuracy": fold_accuracy,
        })
        print(f"  Fold {fold_idx+1} — Val Loss: {avg_fold_val_loss:.4f}, Accuracy: {fold_accuracy:.2f}%")

        # Cleanup per fold
        del model, optimizer, train_loader, val_loader
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- Average across folds ---
    mean_val_loss = float(np.mean(fold_losses))
    mean_accuracy = float(np.mean(fold_accuracies))

    wandb.log({
        "mean_cv_val_loss": mean_val_loss,
        "mean_cv_val_accuracy": mean_accuracy,
    })

    print(f"\n  Trial {current_call} finished — Mean CV Loss: {mean_val_loss:.4f}, Mean Accuracy: {mean_accuracy:.2f}%")
    run.finish()

    # BO minimerer mean_val_loss
    return mean_val_loss

### Checkpoint Logic & Bayesian Optimization

I denne celle håndterer vi både checkpointing og selve kaldet til Bayesian Optimization (BO). Formålet er at kunne genoptage tidligere optimeringskørsler og samtidig sikre, at vi overholder vores fastsatte træningsbudget.

Vi bruger checkpoint-filer til at gemme tidligere evaluerede hyperparametre og deres tilhørende validation loss. Hvis en checkpoint-fil findes, loader vi disse værdier som warm-start data (x0, y0). Det betyder, at BO ikke starter fra scratch, men bygger videre på allerede indsamlet information.

Vi bruger gp_minimize med en Gaussian Process som surrogate model og Expected Improvement (EI) som acquisition function. Parameteren xi=0.01 gør optimeringen relativt eksploitativ, dvs. den fokuserer mere på områder tæt på det bedste fundne punkt.

Budgettet styres via CALLS, som angiver hvor mange nye evalueringer vi ønsker. Hvis vi loader et checkpoint med tidligere trials, reduceres antallet af nye kald tilsvarende.

*Under implementeringen opstod en teknisk constraint fra skopt:*
n_calls skal være større end eller lig med antallet af warm-start punkter plus antallet af initial random points. Derfor justeres n_calls og n_initial_points, så denne betingelse altid overholdes. Denne rettelse ændrer ikke tidligere resultater, men sikrer blot at optimeringen kan fortsætte korrekt.

Denne opsætning er vigtig for sammenligningen mellem BO og grid search, fordi warm-start kan give BO en strukturel fordel. Derfor skal budgetdefinitionen beskrives klart i rapporten: tæller vi kun nye evalueringer, eller total antal modeltræninger?

**Hvad kunne vi ellers have gjort?**
Vi kunne have talt warm-start punkter med i budgettet for en mere strikt fairness mellem BO og grid search. Alternativt kunne vi have defineret antallet af initial random points som en procentdel af budgettet, så balancen mellem exploration og exploitation automatisk tilpasses budgetstørrelsen.

In [10]:
# ==========================================
# 5. Checkpoint Logic & Bayesian Optimization
# ==========================================
if __name__ == '__main__':

    x0 = None
    y0 = None
    current_call = 0
    checkpoint_id_for_this_run = None
    checkpoint_file = None

    # ------------------------------------------------------------
    # A) Checkpoint loading / resume logic
    # ------------------------------------------------------------
    if USE_CHECKPOINT:

        if DESIRED_CHECKPOINT_ID is not None:
            checkpoint_id_for_this_run = DESIRED_CHECKPOINT_ID
            checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'

            if os.path.exists(checkpoint_file):
                print(f"Attempting to load specific checkpoint from {checkpoint_file}...")
                try:
                    res_loaded = load(checkpoint_file)

                    x0 = [list(xi) for xi in res_loaded.x_iters]
                    y0 = list(res_loaded.func_vals)

                    current_call = len(x0)
                    best_so_far = min(y0)

                    print(f"Resuming from {current_call} previous calls from ID {checkpoint_id_for_this_run}.")
                    print(f"  Best loss so far: {best_so_far:.4f}")

                except Exception as e:
                    print(f"WARNING: Could not load checkpoint {checkpoint_file}: {e}. Starting new.")
                    checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
                    checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'
                    print(f"Starting new optimization with checkpoint ID {checkpoint_id_for_this_run}.")
            else:
                print(f"ERROR: Checkpoint file {checkpoint_file} not found. Starting new optimization.")
                checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
                checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'
                print(f"Starting new optimization with checkpoint ID {checkpoint_id_for_this_run}.")

        else:
            latest_id = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=True)

            if latest_id is not None:
                checkpoint_id_for_this_run = latest_id
                checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'

                print(f"Attempting to load latest checkpoint from {checkpoint_file}...")

                try:
                    res_loaded = load(checkpoint_file)

                    x0 = [list(xi) for xi in res_loaded.x_iters]
                    y0 = list(res_loaded.func_vals)

                    current_call = len(x0)
                    best_so_far = min(y0)

                    print(f"Resuming from {current_call} previous calls from latest ID {checkpoint_id_for_this_run}.")
                    print(f"  Best loss so far: {best_so_far:.4f}")

                except Exception as e:
                    print(f"WARNING: Could not load checkpoint {checkpoint_file}: {e}. Starting new.")
                    checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
                    checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'
                    print(f"Starting new optimization with checkpoint ID {checkpoint_id_for_this_run}.")
            else:
                print("No existing checkpoints found. Starting new optimization.")
                checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
                checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'
                print(f"Starting new optimization with checkpoint ID {checkpoint_id_for_this_run}.")

    else:
        print("USE_CHECKPOINT is False. Starting a brand new optimization.")
        checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
        checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'
        print(f"New optimization will use checkpoint ID {checkpoint_id_for_this_run}.")

    if checkpoint_file is None:
        checkpoint_id_for_this_run = get_checkpoint_id(CHECKPOINT_BASE_NAME, find_latest=False)
        checkpoint_file = f'{DRIVE_DIR}/{CHECKPOINT_BASE_NAME}_{checkpoint_id_for_this_run}.pkl'

    checkpoint_callback = CheckpointSaver(checkpoint_file)

    remaining_calls = max(0, CALLS - current_call)

    print(f"Starting optimization with {remaining_calls} remaining calls (Total CALLS: {CALLS})...")
    start_time = time.time()

    # ------------------------------------------------------------
    # B) Bayesian Optimization
    # ------------------------------------------------------------
    if remaining_calls > 0:

        # Select search space
        if USE_NARROWED_SPACE and 'narrowed_search_space' in dir() and narrowed_search_space is not None:
            _active_space = narrowed_search_space

            if x0 is None and narrowed_x0:
                x0 = narrowed_x0
                y0 = narrowed_y0

            print(f"Using NARROWED search space with {len(x0) if x0 else 0} warm-start points.")
        else:
            _active_space = search_space
            print("Using ORIGINAL search space.")

        # --- Initial random exploration logic ---
        if USE_NARROWED_SPACE and x0 is not None and len(x0) > 0:
            required_random = 0
        else:
            required_random = max(0, 20 - len(x0 if x0 is not None else []))

        # =========================================================
        # 🔧 CRITICAL FIX FOR SKOPT CONSTRAINT
        # =========================================================
        n_prev = len(x0) if x0 is not None else 0
        total_calls = n_prev + remaining_calls

        # Ensure n_initial_points never violates skopt constraint
        required_random = min(required_random, total_calls - n_prev)
        # =========================================================

        print(f"Total calls (incl. warm-start): {total_calls}")
        print(f"Initial random points: {required_random}")

        res = gp_minimize(
            train_model,
            _active_space,
            acq_func="EI",
            xi=0.01,
            n_calls=total_calls,
            n_initial_points=required_random,
            noise="gaussian",
            random_state=SEED,
            callback=[checkpoint_callback],
            x0=x0,
            y0=y0,
        )

    else:
        print(f"All {CALLS} calls already completed based on loaded checkpoint.")

        if x0 is not None and y0 is not None:
            best_idx = np.argmin(y0)
            best_lr, best_wd, best_dropout = x0[best_idx]
            best_loss = y0[best_idx]

            class MockResult:
                def __init__(self, x, fun):
                    self.x = x
                    self.fun = fun

            res = MockResult([best_lr, best_wd, best_dropout], best_loss)

            print(f"Best from checkpoint — LR: {best_lr:.6f}, WD: {best_wd:.6f}, "
                  f"Dropout: {best_dropout:.4f}, Loss: {best_loss:.4f}")

    end_time = time.time()

    print(f"\nOptimization finished in {(end_time - start_time)/60:.2f} minutes.")

    if 'res' in locals():
        print(f"Best LR: {res.x[0]:.6f}, "
              f"Best Weight Decay: {res.x[1]:.6f}, "
              f"Best Dropout: {res.x[2]:.4f}, "
              f"Best Loss: {res.fun:.4f}")

USE_CHECKPOINT is False. Starting a brand new optimization.
New optimization will use checkpoint ID 0.
Starting optimization with 18 remaining calls (Total CALLS: 18)...
Using ORIGINAL search space.
Total calls (incl. warm-start): 18
Initial random points: 18


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



  Trial 1/18
  lr=0.024526  wd=0.000036  dropout=0.3898
Using device: cpu

  --- Fold 1/3 ---


KeyboardInterrupt: 

### Analyse af BO checkpoint og indsnævring af søge-rum

I denne celle analyserer vi et tidligere Bayesian Optimization (BO) checkpoint for at finde et mere relevant område i hyperparameter-rummet. Ideen er at bruge de bedste tidligere trials til at definere et “narrowed search space”, som BO senere kan optimere i.

Vi loader den nyeste checkpoint-fil og henter alle tidligere evaluerede punkter (x_iters) og deres losses (func_vals). Derefter ser vi kun på de første N trials og sorterer dem efter mean CV validation loss. Vi vælger de TOP_X bedste og bruger deres minimum og maksimum for learning rate, weight decay og dropout som et udgangspunkt for nye bounds.

For learning rate og weight decay giver det mening at arbejde i log-skala, fordi værdierne typisk varierer over størrelsesordener. Derfor udvider vi bounds i log-space med en margin. For dropout udvider vi i lineær skala. Til sidst clamp’er vi alle nye bounds til de originale grænser, så vi ikke får værdier udenfor det oprindelige search space.

Vi gemmer den nye søge-rumsdefinition i narrowed_search_space, og vi samler også tidligere evaluerede punkter som ligger indenfor de nye bounds i narrowed_x0 og narrowed_y0. De kan bruges til warm-start, så BO starter med relevante datapunkter.

**Hvad kunne vi ellers have gjort?**
Vi kunne have brugt et “blødere” mål end min/max fra TOP_X, fx percentiler (10%-90%) eller en robust statistik, så bounds ikke bliver for påvirket af heldige/noisy trials. Vi kunne også have lavet en tilsvarende “zoom-in” strategi for grid search for at gøre sammenligningen mere fair, fordi narrowed space og warm-start giver BO en ekstra fordel.

In [ ]:
# ==========================================
# 5b. Analyse BO Checkpoint → Narrowed Search Space
# ==========================================
# Load the BO checkpoint, extract the first N trials, find the top X
# by mean_val_loss, compute their min/max per dimension, and define
# a narrowed search space with a configurable margin.
# The results are stored in `narrowed_search_space`, `narrowed_x0`,
# `narrowed_y0` for use by the BO cell (set USE_NARROWED_SPACE = True).

# --- Configuration ---
ANALYSE_FIRST_N_TRIALS = 36   # Only look at the first N trials from the BO checkpoint
TOP_X = 10                    # Number of best trials to base the narrowed space on
MARGIN_PCT = 5                # % margin added above max and below min of each dimension

# OBS: Hvis jeres primære BO-budget er CALLS=18, men I analyserer 36 trials,
# så bygger narrowed space på ekstra historik (evt. flere runs).
# Det kan give BO en strukturel fordel ift. grid search, medmindre I beskriver det tydeligt.

# Original search space bounds (for clamping)
ORIG_BOUNDS = {
    'learning_rate': (1e-4, 1e-1),
    'weight_decay':  (1e-5, 1e-2),
    'dropout':       (0.0,  0.5),
}

# --- Load BO checkpoint ---
_analyse_ckpt_file = None

# OBS: Her hardcodes checkpoint base-name '3d_cv_optimization'.
# Hvis CHECKPOINT_BASE_NAME i BO-cellen er anderledes, kan denne analyse pege på en anden fil.
_analyse_latest_id = get_checkpoint_id('3d_cv_optimization', find_latest=True)

if _analyse_latest_id is not None:
    _analyse_ckpt_file = os.path.join(DRIVE_DIR, f"3d_cv_optimization_{_analyse_latest_id}.pkl")
    print(f"Loading BO checkpoint: {_analyse_ckpt_file}")
else:
    print("ERROR: No BO checkpoint found. Run the BO cell first.")

if _analyse_ckpt_file and os.path.exists(_analyse_ckpt_file):
    _res = load(_analyse_ckpt_file)
    _all_x = [list(xi) for xi in _res.x_iters]
    _all_y = list(_res.func_vals)
    print(f"Total trials in checkpoint: {len(_all_x)}")

    # Restrict to first N trials
    _n = min(ANALYSE_FIRST_N_TRIALS, len(_all_x))
    _x_subset = _all_x[:_n]
    _y_subset = _all_y[:_n]
    print(f"Analysing first {_n} trials.")

    # Rank by loss and take top X
    # OBS: Hvis objective er noisy (SGD + CV), kan "top X" indeholde heldige runs.
    # Så bounds kan blive for snævre. Margin hjælper, men valg af TOP_X er vigtigt.
    _ranked = sorted(zip(_y_subset, _x_subset, range(1, _n + 1)), key=lambda t: t[0])
    _top = _ranked[:TOP_X]

    print(f"\n{'='*74}")
    print(f"  Top {len(_top)} trials (out of first {_n}) by mean CV val loss")
    print(f"{'='*74}")
    print(f"  {'Rank':<5} {'Trial#':<8} {'Loss':<10} {'LR':<14} {'WD':<14} {'Dropout':<10}")
    print(f"  {'-'*5} {'-'*8} {'-'*10} {'-'*14} {'-'*14} {'-'*10}")
    for i, (loss, params, trial_num) in enumerate(_top):
        print(f"  {i+1:<5} {trial_num:<8} {loss:<10.4f} {params[0]:<14.6e} {params[1]:<14.6e} {params[2]:<10.4f}")

    # Extract per-dimension min/max from top X
    _top_params = [p for _, p, _ in _top]
    _top_lrs  = [p[0] for p in _top_params]
    _top_wds  = [p[1] for p in _top_params]
    _top_dos  = [p[2] for p in _top_params]

    _raw_bounds = {
        'learning_rate': (min(_top_lrs), max(_top_lrs)),
        'weight_decay':  (min(_top_wds), max(_top_wds)),
        'dropout':       (min(_top_dos), max(_top_dos)),
    }

    print(f"\n  Raw bounds from top {TOP_X}:")
    for dim, (lo, hi) in _raw_bounds.items():
        scale = "log" if dim != "dropout" else "lin"
        print(f"    {dim:<16} [{lo:.6e}, {hi:.6e}]  ({scale})")

    # Apply margin — for log-uniform dims, margin is applied in log-space
    margin_frac = MARGIN_PCT / 100.0

    def _apply_margin(lo, hi, orig_lo, orig_hi, is_log=False):
        """Expand [lo, hi] by margin_frac, clamped to original bounds."""
        if is_log:
            log_lo, log_hi = np.log10(lo), np.log10(hi)
            log_range = log_hi - log_lo
            log_range = max(log_range, 0.1)  # min 0.1 decades
            new_log_lo = log_lo - margin_frac * log_range
            new_log_hi = log_hi + margin_frac * log_range
            new_lo = max(10 ** new_log_lo, orig_lo)
            new_hi = min(10 ** new_log_hi, orig_hi)
        else:
            lin_range = hi - lo
            lin_range = max(lin_range, 0.02)  # min range 0.02
            new_lo = max(lo - margin_frac * lin_range, orig_lo)
            new_hi = min(hi + margin_frac * lin_range, orig_hi)
        return new_lo, new_hi

    _narrowed_lr = _apply_margin(*_raw_bounds['learning_rate'], *ORIG_BOUNDS['learning_rate'], is_log=True)
    _narrowed_wd = _apply_margin(*_raw_bounds['weight_decay'],  *ORIG_BOUNDS['weight_decay'],  is_log=True)
    _narrowed_do = _apply_margin(*_raw_bounds['dropout'],       *ORIG_BOUNDS['dropout'],       is_log=False)

    # --- Compare narrowed vs original bounds ---
    def _pct_of_original(new_lo, new_hi, orig_lo, orig_hi, is_log=False):
        """Compute what % of the original range the narrowed range covers."""
        if is_log:
            orig_range = np.log10(orig_hi) - np.log10(orig_lo)
            new_range  = np.log10(new_hi)  - np.log10(new_lo)
        else:
            orig_range = orig_hi - orig_lo
            new_range  = new_hi  - new_lo
        return 100.0 * new_range / orig_range if orig_range > 0 else 0.0

    _pct_lr = _pct_of_original(*_narrowed_lr, *ORIG_BOUNDS['learning_rate'], is_log=True)
    _pct_wd = _pct_of_original(*_narrowed_wd, *ORIG_BOUNDS['weight_decay'],  is_log=True)
    _pct_do = _pct_of_original(*_narrowed_do, *ORIG_BOUNDS['dropout'],       is_log=False)

    print(f"\n  Narrowed bounds (with {MARGIN_PCT}% margin, clamped to original space):")
    print(f"  {'Dimension':<16} {'Narrowed Range':<36} {'Original Range':<36} {'% of Original':<14}")
    print(f"  {'-'*16} {'-'*36} {'-'*36} {'-'*14}")
    print(f"  {'learning_rate':<16} [{_narrowed_lr[0]:.6e}, {_narrowed_lr[1]:.6e}]  (log)   "
          f"[{ORIG_BOUNDS['learning_rate'][0]:.6e}, {ORIG_BOUNDS['learning_rate'][1]:.6e}]  (log)   "
          f"{_pct_lr:>6.1f}%")
    print(f"  {'weight_decay':<16} [{_narrowed_wd[0]:.6e}, {_narrowed_wd[1]:.6e}]  (log)   "
          f"[{ORIG_BOUNDS['weight_decay'][0]:.6e}, {ORIG_BOUNDS['weight_decay'][1]:.6e}]  (log)   "
          f"{_pct_wd:>6.1f}%")
    print(f"  {'dropout':<16} [{_narrowed_do[0]:.6e}, {_narrowed_do[1]:.6e}]  (lin)   "
          f"[{ORIG_BOUNDS['dropout'][0]:.6e}, {ORIG_BOUNDS['dropout'][1]:.6e}]  (lin)   "
          f"{_pct_do:>6.1f}%")

    # OBS: Denne volume-beregning er kun til rapport/print.
    # Hvis man vil have "procent af volumen" korrekt, bør man normalt dividere med 100^3,
    # men her divideres med 100^2, så tallet bliver skaleret forkert.
    _total_vol_pct = _pct_lr * _pct_wd * _pct_do / (100 * 100)
    print(f"\n  Combined volume: {_total_vol_pct:.1f}% of original search space")

    # --- Build the narrowed search space (same format as the original) ---
    narrowed_search_space = [
        Real(_narrowed_lr[0], _narrowed_lr[1], prior='log-uniform', name='learning_rate'),
        Real(_narrowed_wd[0], _narrowed_wd[1], prior='log-uniform', name='weight_decay'),
        Real(_narrowed_do[0], _narrowed_do[1], prior='uniform',     name='dropout'),
    ]

    # Collect evaluated points that fall within the narrowed bounds (for warm-starting BO)
    narrowed_x0 = []
    narrowed_y0 = []
    for xi, yi in zip(_x_subset, _y_subset):
        lr_ok = _narrowed_lr[0] <= xi[0] <= _narrowed_lr[1]
        wd_ok = _narrowed_wd[0] <= xi[1] <= _narrowed_wd[1]
        do_ok = _narrowed_do[0] <= xi[2] <= _narrowed_do[1]
        if lr_ok and wd_ok and do_ok:
            narrowed_x0.append(xi)
            narrowed_y0.append(yi)

    print(f"\n  Warm-start points inside narrowed space: {len(narrowed_x0)} / {_n}")
    if narrowed_x0:
        _best_ws_idx = int(np.argmin(narrowed_y0))
        print(f"  Best warm-start point: loss={narrowed_y0[_best_ws_idx]:.4f}")

    print(f"\n  To use in the BO cell, set USE_NARROWED_SPACE = True")
    print(f"  Variables available: narrowed_search_space, narrowed_x0, narrowed_y0")

    # Cleanup
    del _res, _all_x, _all_y, _x_subset, _y_subset, _ranked, _top, _top_params
    del _top_lrs, _top_wds, _top_dos, _raw_bounds

else:
    if _analyse_ckpt_file:
        print(f"ERROR: Checkpoint file {_analyse_ckpt_file} not found.")
    narrowed_search_space = None
    narrowed_x0 = None
    narrowed_y0 = None

### 3D visualisering af BO-checkpoint

I denne celle laver vi en interaktiv 3D visualisering af resultaterne fra Bayesian Optimization. Formålet er at få et visuelt overblik over, hvor i hyperparameter-rummet BO har prøvet forskellige kombinationer, og hvordan loss varierer i forhold til learning rate og weight decay.

Vi loader den nyeste checkpoint-fil og henter alle tidligere prøvede hyperparametre (x_iters) samt deres tilhørende mean CV validation loss (func_vals). Hvis variablen ANALYSE_FIRST_N_TRIALS findes fra en tidligere analyse-celle, begrænser vi plottet til de første N trials for at matche den analyse.

Vi plotter learning rate og weight decay på log10-skala, fordi de varierer over størrelsesordener. Loss plottes på z-aksen. Farven på hver markør repræsenterer loss (lavere er bedre), og dropout er kodet som markørstørrelse, så højere dropout bliver vist som større markører. De bedste TOP_X trials markeres separat, så man kan se hvor de ligger i rummet.

Hvis vi har beregnet et “narrowed search space” i analyse-cellen, tegner vi også en boks i plottet, som viser de nye indsnævrede grænser. På den måde kan man visuelt vurdere om narrowed space faktisk dækker det område, hvor de bedste trials ligger.

**Hvad kunne vi ellers have gjort?**
Vi kunne have lavet flere plots der isolerer én parameter ad gangen, fx 2D scatter plots (LR vs loss og WD vs loss) eller et plot hvor dropout ligger på en separat akse i stedet for markørstørrelse. Vi kunne også have plottet alle trials uden log-skala og i stedet vist akserne i scientific notation, men det gør ofte mønstre i LR/WD sværere at se.

In [ ]:
# ==========================================
# 5c. 3D Visualisation of BO Checkpoint
# ==========================================
# Interactive 3D scatter plot: LR vs WD vs Loss, coloured by loss.
# Dropout is encoded as marker size (larger = higher dropout).
# The top-X trials from the analysis cell are highlighted in red.

import plotly.graph_objects as go

# --- Load checkpoint (reuse the same logic as 5b) ---
# OBS: checkpoint base-name er hardcoded til '3d_cv_optimization'
# Hvis jeres BO-run gemmer under et andet base-name, kan I ende med at visualisere "forkert" run.
_viz_ckpt_id = get_checkpoint_id('3d_cv_optimization', find_latest=True)
if _viz_ckpt_id is None:
    raise RuntimeError("No BO checkpoint found. Run the BO cell first.")

_viz_ckpt_file = os.path.join(DRIVE_DIR, f"3d_cv_optimization_{_viz_ckpt_id}.pkl")
_viz_res = load(_viz_ckpt_file)

# x_iters: tidligere evaluerede hyperparametre (lr, wd, dropout)
# func_vals: den objective vi optimerer (mean CV val loss)
_viz_x = [list(xi) for xi in _viz_res.x_iters]
_viz_y = list(_viz_res.func_vals)

# Restrict to analysed subset if the variable exists
# OBS: 'dir()' i notebooks afhænger af hvilke celler der er kørt.
# Hvis ANALYSE_FIRST_N_TRIALS ikke findes, bruges alle trials i checkpointet.
_viz_n = min(ANALYSE_FIRST_N_TRIALS, len(_viz_x)) if 'ANALYSE_FIRST_N_TRIALS' in dir() else len(_viz_x)
_viz_x = _viz_x[:_viz_n]
_viz_y = _viz_y[:_viz_n]

# Pak parametrene ud i separate arrays for plotting
_viz_lr  = np.array([p[0] for p in _viz_x])
_viz_wd  = np.array([p[1] for p in _viz_x])
_viz_do  = np.array([p[2] for p in _viz_x])
_viz_loss = np.array(_viz_y)

# Scale dropout → marker size (min 4, max 18)
# Dvs. højere dropout = større markør i plottet.
_do_min, _do_max = _viz_do.min(), _viz_do.max()
if _do_max > _do_min:
    _viz_sizes = 4 + 14 * (_viz_do - _do_min) / (_do_max - _do_min)
else:
    _viz_sizes = np.full_like(_viz_do, 10.0)

# Identify top-X indices (same TOP_X as 5b)
# OBS: Top-X defineres kun ud fra loss (lavest = bedst).
_top_x_count = TOP_X if 'TOP_X' in dir() else 5
_sorted_idx = np.argsort(_viz_loss)
_top_idx = set(_sorted_idx[:_top_x_count])
_is_top = np.array([i in _top_idx for i in range(len(_viz_loss))])

# Hover text for interaktivt plot (viser trial nr og parametre)
_hover = [
    f"Trial {i+1}<br>LR: {_viz_lr[i]:.6e}<br>WD: {_viz_wd[i]:.6e}<br>"
    f"Dropout: {_viz_do[i]:.4f}<br>Loss: {_viz_loss[i]:.4f}"
    for i in range(len(_viz_loss))
]

# --- Build figure ---
fig = go.Figure()

# All trials (ikke-top)
# OBS: LR og WD plottes i log10, så aksen er log-skala.
fig.add_trace(go.Scatter3d(
    x=np.log10(_viz_lr[~_is_top]),
    y=np.log10(_viz_wd[~_is_top]),
    z=_viz_loss[~_is_top],
    mode='markers',
    marker=dict(
        size=_viz_sizes[~_is_top],
        color=_viz_loss[~_is_top],
        colorscale='Viridis',
        colorbar=dict(title='Loss', x=1.05),
        opacity=0.6,
        line=dict(width=0.5, color='white'),
    ),
    text=[h for h, t in zip(_hover, _is_top) if not t],
    hoverinfo='text',
    name='Trials',
))

# Top X trials (highlighted)
fig.add_trace(go.Scatter3d(
    x=np.log10(_viz_lr[_is_top]),
    y=np.log10(_viz_wd[_is_top]),
    z=_viz_loss[_is_top],
    mode='markers',
    marker=dict(
        size=_viz_sizes[_is_top] + 4,
        color='red',
        opacity=0.9,
        symbol='diamond',
        line=dict(width=1, color='darkred'),
    ),
    text=[h for h, t in zip(_hover, _is_top) if t],
    hoverinfo='text',
    name=f'Top {_top_x_count}',
))

# If narrowed bounds exist, draw the narrowed bounding box
if 'narrowed_search_space' in dir() and narrowed_search_space is not None:
    # OBS: Her bruges _narrowed_lr/_narrowed_wd, som forventes sat i 5b-cellen.
    _nb_lr = [np.log10(_narrowed_lr[0]), np.log10(_narrowed_lr[1])]
    _nb_wd = [np.log10(_narrowed_wd[0]), np.log10(_narrowed_wd[1])]
    _nb_z_lo = float(_viz_loss.min()) - 0.01
    _nb_z_hi = float(_viz_loss.max()) + 0.01

    # 12 edges of a rectangular box
    def _box_edges(x0, x1, y0, y1, z0, z1):
        edges_x, edges_y, edges_z = [], [], []
        for (xa, ya, za), (xb, yb, zb) in [
            ((x0,y0,z0),(x1,y0,z0)), ((x0,y1,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x1,y0,z1)), ((x0,y1,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y1,z0)), ((x1,y0,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x0,y1,z1)), ((x1,y0,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y0,z1)), ((x1,y0,z0),(x1,y0,z1)),
            ((x0,y1,z0),(x0,y1,z1)), ((x1,y1,z0),(x1,y1,z1)),
        ]:
            edges_x += [xa, xb, None]
            edges_y += [ya, yb, None]
            edges_z += [za, zb, None]
        return edges_x, edges_y, edges_z

    bx, by, bz = _box_edges(_nb_lr[0], _nb_lr[1], _nb_wd[0], _nb_wd[1], _nb_z_lo, _nb_z_hi)
    fig.add_trace(go.Scatter3d(
        x=bx, y=by, z=bz,
        mode='lines',
        line=dict(color='orange', width=3),
        name='Narrowed bounds',
        hoverinfo='skip',
    ))

# Layout: aksetitler og størrelse
fig.update_layout(
    title=f'BO Trials (first {_viz_n}) — Marker size ∝ Dropout',
    scene=dict(
        xaxis_title='log₁₀(Learning Rate)',
        yaxis_title='log₁₀(Weight Decay)',
        zaxis_title='Mean CV Val Loss',
    ),
    width=900,
    height=700,
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

# Cleanup
del _viz_res, _viz_x, _viz_y, _viz_lr, _viz_wd, _viz_do, _viz_loss
del _viz_sizes, _sorted_idx, _top_idx, _is_top, _hover

### Grid search (baseline) med checkpoint og evt. indsnævret grid

I denne celle kører vi grid search som baseline til at sammenligne med Bayesian Optimization. Grid search betyder, at vi vælger et fast sæt værdier for learning rate, weight decay og dropout og evaluerer alle kombinationer (full-factorial search). Learning rate og weight decay er log-spaced, mens dropout er lineært spaced.

Cellen kan køre i to modes. Hvis GRID_USE_NARROWED_SPACE=True og variablen narrowed_grid_space findes fra analyse-cellen (6b), bruger vi de indsnævrede bounds og den opløsning der er gemt dér. Ellers bruger vi de originale bounds og opløsningen angivet af GRID_N_LR, GRID_N_WD og GRID_N_DO.

For at kunne genoptage grid search efter afbrydelser gemmer vi resultater efter hver trial i en JSON checkpoint-fil. Når cellen starter, forsøger den at loade den nyeste checkpoint og springer allerede gennemførte kombinationer over, så vi kun kører de resterende.

Vi genbruger den samme train_model(params) funktion som i BO, så træning og evaluering er identisk mellem metoderne (samme CV, epochs, batch size osv.). For at holde WandB-logning samlet under grid search ændrer vi midlertidigt nogle globale variabler, og til sidst restore vi dem igen.

**Hvad kunne vi ellers have gjort?**
Vi kunne have brugt random search som baseline, fordi den ofte er stærkere end et groft grid ved samme budget. Vi kunne også have undgået at ændre globale variabler ved at lade train_model tage et argument for “log group”, så notebook-state bliver mindre skrøbelig hvis der opstår errors midt i grid search.

In [ ]:
# ==========================================
# 6. Grid Search Baseline
# ==========================================
# This cell runs a full-factorial grid search over the same 3D
# hyperparameter space used by BO, to serve as a comparison baseline.
# LR and WD are log-spaced; dropout is linearly spaced.
# Results are checkpointed to a JSON file after each trial for resume support.
# All trials are grouped together in WandB under "grid_search_{id}".

import itertools
import json as _json  # alias to avoid shadowing the earlier import


# --------------------------------------------------
# --- Grid Search Configuration ---
# --------------------------------------------------

GRID_CHECKPOINT_BASE_NAME = 'grid_search'

# If True: try to resume from existing JSON checkpoint
GRID_USE_CHECKPOINT = True  

# If True: use narrowed bounds from cell 6b (if available)
GRID_USE_NARROWED_SPACE = True  

# Resolution of ORIGINAL grid (used only if narrowed grid is NOT active)
GRID_N_LR = 4
GRID_N_WD = 3
GRID_N_DO = 3


# --------------------------------------------------
# --- Build the 3D grid ---
# --------------------------------------------------

# If narrowed grid exists and flag is enabled → use narrowed bounds
if GRID_USE_NARROWED_SPACE and 'narrowed_grid_space' in dir() and narrowed_grid_space is not None:

    # Resolution from narrowed grid (fallback to defaults if missing)
    _n_lr = narrowed_grid_space.get('n_lr', GRID_N_LR)
    _n_wd = narrowed_grid_space.get('n_wd', GRID_N_WD)
    _n_do = narrowed_grid_space.get('n_do', GRID_N_DO)

    # Build grids using narrowed bounds
    lr_grid = np.logspace(
        np.log10(narrowed_grid_space['lr'][0]),
        np.log10(narrowed_grid_space['lr'][1]),
        _n_lr
    ).tolist()

    wd_grid = np.logspace(
        np.log10(narrowed_grid_space['wd'][0]),
        np.log10(narrowed_grid_space['wd'][1]),
        _n_wd
    ).tolist()

    dropout_grid = np.linspace(
        narrowed_grid_space['do'][0],
        narrowed_grid_space['do'][1],
        _n_do
    ).tolist()

    # Use separate checkpoint base-name for narrowed grid
    GRID_CHECKPOINT_BASE_NAME = 'grid_search_narrowed'

    print(f"Using NARROWED grid bounds from cell 6b ({_n_lr}×{_n_wd}×{_n_do}).")

else:
    # ORIGINAL search space bounds
    lr_grid = np.logspace(np.log10(1e-4), np.log10(1e-1), GRID_N_LR).tolist()
    wd_grid = np.logspace(np.log10(1e-5), np.log10(1e-2), GRID_N_WD).tolist()
    dropout_grid = np.linspace(0.0, 0.5, GRID_N_DO).tolist()

    if GRID_USE_NARROWED_SPACE:
        print("WARNING: GRID_USE_NARROWED_SPACE=True but narrowed_grid_space not found. Using original.")

    print("Using ORIGINAL grid bounds.")


# Total number of grid combinations
GRID_TOTAL = len(lr_grid) * len(wd_grid) * len(dropout_grid)


# Generate all hyperparameter combinations in deterministic order
grid_combinations = [
    [lr, wd, do]
    for lr, wd, do in itertools.product(lr_grid, wd_grid, dropout_grid)
]

print(f"Grid Search: {len(lr_grid)} LR × {len(wd_grid)} WD × {len(dropout_grid)} Dropout = {GRID_TOTAL} trials")
print(f"  LR grid  (log): {[f'{v:.6f}' for v in lr_grid]}")
print(f"  WD grid  (log): {[f'{v:.6f}' for v in wd_grid]}")
print(f"  Dropout  (lin): {[f'{v:.4f}' for v in dropout_grid]}")


# --------------------------------------------------
# --- JSON checkpoint helpers ---
# --------------------------------------------------

def _get_grid_checkpoint_id(base_name, find_latest=False):
    """Find existing grid search JSON checkpoint IDs in DRIVE_DIR."""
    existing_ids = []
    for f_name in os.listdir(DRIVE_DIR):
        match = re.match(rf'^{re.escape(base_name)}_(\d+)\.json$', f_name)
        if match:
            existing_ids.append(int(match.group(1)))

    if find_latest:
        return max(existing_ids) if existing_ids else None
    else:
        if not existing_ids:
            return 0
        existing_ids.sort()
        for i, _id in enumerate(existing_ids):
            if i != _id:
                return i
        return len(existing_ids)


def _load_grid_checkpoint(filepath):
    """Load completed results from JSON checkpoint."""
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            data = _json.load(f)
        return data.get("results", [])
    return []


def _save_grid_checkpoint(filepath, results, grid):
    """Save results and full grid definition to JSON checkpoint."""
    with open(filepath, "w") as f:
        _json.dump({"grid": grid, "results": results}, f, indent=2)


# --------------------------------------------------
# --- Resolve checkpoint (resume if possible) ---
# --------------------------------------------------

grid_checkpoint_id = None
grid_checkpoint_file = None
grid_completed_results = []

if GRID_USE_CHECKPOINT:

    latest_id = _get_grid_checkpoint_id(GRID_CHECKPOINT_BASE_NAME, find_latest=True)

    if latest_id is not None:
        grid_checkpoint_id = latest_id
        grid_checkpoint_file = os.path.join(
            DRIVE_DIR,
            f"{GRID_CHECKPOINT_BASE_NAME}_{grid_checkpoint_id}.json"
        )

        grid_completed_results = _load_grid_checkpoint(grid_checkpoint_file)

        print(f"Loaded grid checkpoint ID {grid_checkpoint_id} with {len(grid_completed_results)} completed trials.")

        if grid_completed_results:
            best_prev = min(grid_completed_results, key=lambda r: r["mean_val_loss"])
            print(f"  Best so far: loss={best_prev['mean_val_loss']:.4f} "
                  f"(lr={best_prev['params'][0]:.6f}, wd={best_prev['params'][1]:.6f}, do={best_prev['params'][2]:.4f})")

    else:
        print("No existing grid search checkpoints found. Starting new.")


# If no checkpoint was found → create new ID
if grid_checkpoint_id is None:
    grid_checkpoint_id = _get_grid_checkpoint_id(GRID_CHECKPOINT_BASE_NAME, find_latest=False)
    grid_checkpoint_file = os.path.join(
        DRIVE_DIR,
        f"{GRID_CHECKPOINT_BASE_NAME}_{grid_checkpoint_id}.json"
    )
    print(f"New grid search will use checkpoint ID {grid_checkpoint_id}.")


# --------------------------------------------------
# --- Prepare resume logic ---
# --------------------------------------------------

# Build set of already-completed param combinations (rounded to avoid float issues)
_completed_set = set()
for r in grid_completed_results:
    _completed_set.add(tuple(round(v, 10) for v in r["params"]))


# --------------------------------------------------
# --- Override globals for WandB grouping ---
# --------------------------------------------------

_saved_CHECKPOINT_BASE_NAME = CHECKPOINT_BASE_NAME
_saved_checkpoint_id = checkpoint_id_for_this_run
_saved_current_call = current_call
_saved_CALLS = CALLS

CHECKPOINT_BASE_NAME = GRID_CHECKPOINT_BASE_NAME
checkpoint_id_for_this_run = grid_checkpoint_id
current_call = len(grid_completed_results)
CALLS = GRID_TOTAL


# --------------------------------------------------
# --- Main grid search loop ---
# --------------------------------------------------

remaining_grid = [
    combo for combo in grid_combinations
    if tuple(round(v, 10) for v in combo) not in _completed_set
]

print(f"\n{'='*60}")
print(f"  Grid Search: {len(remaining_grid)} remaining / {GRID_TOTAL} total trials")
print(f"{'='*60}")

grid_start_time = time.time()
best_grid_loss = min((r["mean_val_loss"] for r in grid_completed_results), default=float("inf"))

for combo_idx, params in enumerate(remaining_grid):

    lr, wd, do = params
    trial_num = current_call + 1

    print(f"\n>>> Grid trial {trial_num}/{GRID_TOTAL}  "
          f"[lr={lr:.6f}, wd={wd:.6f}, dropout={do:.4f}]")

    # Reuse the exact same training function as BO
    mean_val_loss = train_model(params)

    # Record result
    result_entry = {
        "params": [lr, wd, do],
        "mean_val_loss": mean_val_loss,
        "trial": trial_num,
    }

    grid_completed_results.append(result_entry)

    # Track best result
    if mean_val_loss < best_grid_loss:
        best_grid_loss = mean_val_loss
        print(f"  *** New best grid loss: {best_grid_loss:.4f}")

    # Save checkpoint after each trial
    _save_grid_checkpoint(grid_checkpoint_file, grid_completed_results, grid_combinations)
    print(f"  Checkpoint saved ({len(grid_completed_results)}/{GRID_TOTAL} done).")


grid_end_time = time.time()


# --------------------------------------------------
# --- Summary ---
# --------------------------------------------------

print(f"\n{'='*60}")
print(f"  Grid Search Complete!")
print(f"  Total time: {(grid_end_time - grid_start_time)/60:.2f} minutes")
print(f"  Trials run this session: {len(remaining_grid)}")
print(f"  Total completed: {len(grid_completed_results)}/{GRID_TOTAL}")
print(f"{'='*60}")

if grid_completed_results:

    best = min(grid_completed_results, key=lambda r: r["mean_val_loss"])

    print(f"\n  Best Grid Search Result:")
    print(f"    Learning Rate: {best['params'][0]:.6f}")
    print(f"    Weight Decay:  {best['params'][1]:.6f}")
    print(f"    Dropout:       {best['params'][2]:.4f}")
    print(f"    Mean CV Loss:  {best['mean_val_loss']:.4f}")

    print(f"\n  All results (sorted by loss):")
    for i, r in enumerate(sorted(grid_completed_results, key=lambda r: r["mean_val_loss"])):
        print(f"    {i+1:2d}. loss={r['mean_val_loss']:.4f}  "
              f"lr={r['params'][0]:.6f}  wd={r['params'][1]:.6f}  do={r['params'][2]:.4f}")


# --------------------------------------------------
# --- Restore globals ---
# --------------------------------------------------

CHECKPOINT_BASE_NAME = _saved_CHECKPOINT_BASE_NAME
checkpoint_id_for_this_run = _saved_checkpoint_id
current_call = _saved_current_call
CALLS = _saved_CALLS

### Analyse af grid search checkpoint og indsnævring af grid-rum

I denne celle analyserer vi resultaterne fra en tidligere grid search for at definere et indsnævret område i hyperparameter-rummet. Ideen er at bruge de bedste grid-trials til at finde et interval for learning rate, weight decay og dropout, som senere kan bruges til en “zoom-in” grid search.

Vi loader den nyeste grid search checkpoint-fil (JSON) og henter listen af tidligere trials med deres hyperparametre og mean CV validation loss. Vi analyserer op til de første N trials og sorterer dem efter loss. Derefter vælger vi de TOP_X bedste og tager min/max for hver dimension som rå bounds.

For learning rate og weight decay udvider vi bounds i log-skala med en margin, fordi disse hyperparametre varierer over størrelsesordener. For dropout udvider vi bounds i lineær skala. Til sidst clamp’er vi de nye bounds til de originale grænser, så vi ikke bevæger os udenfor det oprindelige search space.

Vi gemmer de indsnævrede bounds og en ønsket grid-opløsning i narrowed_grid_space. Til sidst printer vi et preview af, hvilke værdier et narrowed grid vil indeholde, så vi kan se hvor mange trials det giver og hvilke punkter der faktisk køres.

**Hvad kunne vi ellers have gjort?**
I stedet for min/max fra TOP_X kunne vi bruge percentiler eller en robust metode, så bounds ikke bliver for følsomme overfor enkelte støjende/heldige trials. Vi kunne også ændre opløsningen i det indsnævrede grid (fx 5×5×3) for at få en mere finmasket søgning, men det ville øge compute-budgettet.

In [ ]:
# ==========================================
# 6b. Analyse Grid Search Checkpoint → Narrowed Grid Space
# ==========================================
# Formål:
# - Læs et tidligere grid search checkpoint (JSON)
# - Find de bedste trials (lavest mean_val_loss)
# - Brug deres min/max pr. dimension til at definere et "narrowed" søge-rum
# - Udvid bounds med en margin og clamp til original bounds
# - Gem resultatet i `narrowed_grid_space`, som grid-search cellen kan bruge

import json as _json_6b

# --------------------------------------------------
# --- Configuration ---
# --------------------------------------------------

# Vi analyserer kun de første N trials i checkpointet (hvis der er færre, bruges alle).
GRID_ANALYSE_FIRST_N_TRIALS = 36

# Hvor mange af de bedste trials vi baserer narrowed bounds på.
GRID_TOP_X = 10

# Margin (%) som udvider de rå min/max bounds.
GRID_MARGIN_PCT = 5

# Opløsning i det indsnævrede grid (bruges til preview og gemmes i narrowed_grid_space).
# OBS: Dette bestemmer hvor mange trials et senere narrowed-grid kommer til at have.
GRID_N_LR = 3
GRID_N_WD = 3
GRID_N_DO = 2

# Originale bounds (bruges til at clamp'e narrowed bounds, så vi ikke går udenfor).
GRID_ORIG_BOUNDS = {
    'learning_rate': (1e-4, 1e-1),
    'weight_decay':  (1e-5, 1e-2),
    'dropout':       (0.0,  0.5),
}

# --------------------------------------------------
# --- Load grid search checkpoint ---
# --------------------------------------------------

_gs_ckpt_file = None

# Find nyeste grid checkpoint ID i DRIVE_DIR.
# OBS: base_name er hardcoded til 'grid_search' og matcher grid-cellen.
_gs_latest_id = _get_grid_checkpoint_id('grid_search', find_latest=True)

if _gs_latest_id is not None:
    _gs_ckpt_file = os.path.join(DRIVE_DIR, f"grid_search_{_gs_latest_id}.json")
    print(f"Loading grid search checkpoint: {_gs_ckpt_file}")
else:
    print("ERROR: No grid search checkpoint found. Run the grid search cell first.")

# Fortsæt kun hvis filen faktisk eksisterer
if _gs_ckpt_file and os.path.exists(_gs_ckpt_file):

    # Load JSON
    with open(_gs_ckpt_file, "r") as f:
        _gs_data = _json_6b.load(f)

    # Resultaterne ligger i "results" som en liste af dicts
    _gs_results = _gs_data.get("results", [])
    print(f"Total trials in checkpoint: {len(_gs_results)}")

    # --------------------------------------------------
    # --- Restrict to first N trials ---
    # --------------------------------------------------

    # Vi analyserer maks de første GRID_ANALYSE_FIRST_N_TRIALS (ellers alle).
    _gs_n = min(GRID_ANALYSE_FIRST_N_TRIALS, len(_gs_results))
    _gs_subset = _gs_results[:_gs_n]
    print(f"Analysing first {_gs_n} trials.")

    # --------------------------------------------------
    # --- Rank by loss and take top X ---
    # --------------------------------------------------

    # Sortér efter mean_val_loss (lavest er bedst)
    _gs_ranked = sorted(_gs_subset, key=lambda r: r["mean_val_loss"])
    _gs_top = _gs_ranked[:GRID_TOP_X]

    # Print top-X tabel
    print(f"\n{'='*74}")
    print(f"  Top {len(_gs_top)} grid trials (out of first {_gs_n}) by mean CV val loss")
    print(f"{'='*74}")
    print(f"  {'Rank':<5} {'Trial#':<8} {'Loss':<10} {'LR':<14} {'WD':<14} {'Dropout':<10}")
    print(f"  {'-'*5} {'-'*8} {'-'*10} {'-'*14} {'-'*14} {'-'*10}")

    for i, r in enumerate(_gs_top):
        p = r["params"]  # [lr, wd, dropout]
        # trial nummer findes som r["trial"] i jeres grid-celle, men vi bruger get(...) for robusthed
        print(f"  {i+1:<5} {r.get('trial', '?'):<8} {r['mean_val_loss']:<10.4f} "
              f"{p[0]:<14.6e} {p[1]:<14.6e} {p[2]:<10.4f}")

    # --------------------------------------------------
    # --- Extract raw bounds from top X ---
    # --------------------------------------------------

    # Udpak top parametre dimension for dimension
    _gs_top_lrs = [r["params"][0] for r in _gs_top]
    _gs_top_wds = [r["params"][1] for r in _gs_top]
    _gs_top_dos = [r["params"][2] for r in _gs_top]

    # Rå bounds = min/max blandt top-X
    _gs_raw_bounds = {
        'learning_rate': (min(_gs_top_lrs), max(_gs_top_lrs)),
        'weight_decay':  (min(_gs_top_wds), max(_gs_top_wds)),
        'dropout':       (min(_gs_top_dos), max(_gs_top_dos)),
    }

    print(f"\n  Raw bounds from top {GRID_TOP_X}:")
    for dim, (lo, hi) in _gs_raw_bounds.items():
        scale = "log" if dim != "dropout" else "lin"
        print(f"    {dim:<16} [{lo:.6e}, {hi:.6e}]  ({scale})")

    # --------------------------------------------------
    # --- Apply margin (log-space for LR/WD) ---
    # --------------------------------------------------

    _gs_margin_frac = GRID_MARGIN_PCT / 100.0

    def _gs_apply_margin(lo, hi, orig_lo, orig_hi, is_log=False):
        """
        Udvider intervallet [lo, hi] med en margin og clamp'er til [orig_lo, orig_hi].
        - Hvis is_log=True udvider vi i log10-space (giver mening for LR/WD).
        - Ellers udvider vi lineært (giver mening for dropout).
        """
        if is_log:
            log_lo, log_hi = np.log10(lo), np.log10(hi)
            log_range = log_hi - log_lo

            # Min range (0.1 decades) for at undgå alt for snævre intervaller
            log_range = max(log_range, 0.1)

            new_log_lo = log_lo - _gs_margin_frac * log_range
            new_log_hi = log_hi + _gs_margin_frac * log_range

            new_lo = max(10 ** new_log_lo, orig_lo)
            new_hi = min(10 ** new_log_hi, orig_hi)
        else:
            lin_range = hi - lo

            # Min range for at undgå alt for snæver dropout-range
            lin_range = max(lin_range, 0.02)

            new_lo = max(lo - _gs_margin_frac * lin_range, orig_lo)
            new_hi = min(hi + _gs_margin_frac * lin_range, orig_hi)

        return new_lo, new_hi

    # Udvid bounds for hver dimension og clamp til original bounds
    _gs_narrowed_lr = _gs_apply_margin(
        *_gs_raw_bounds['learning_rate'],
        *GRID_ORIG_BOUNDS['learning_rate'],
        is_log=True
    )
    _gs_narrowed_wd = _gs_apply_margin(
        *_gs_raw_bounds['weight_decay'],
        *GRID_ORIG_BOUNDS['weight_decay'],
        is_log=True
    )
    _gs_narrowed_do = _gs_apply_margin(
        *_gs_raw_bounds['dropout'],
        *GRID_ORIG_BOUNDS['dropout'],
        is_log=False
    )

    # --------------------------------------------------
    # --- Compare narrowed vs original bounds (informativt print) ---
    # --------------------------------------------------

    def _gs_pct_of_original(new_lo, new_hi, orig_lo, orig_hi, is_log=False):
        """Returnér hvor stor en del (%) af original-range den narrowed range dækker."""
        if is_log:
            orig_range = np.log10(orig_hi) - np.log10(orig_lo)
            new_range  = np.log10(new_hi)  - np.log10(new_lo)
        else:
            orig_range = orig_hi - orig_lo
            new_range  = new_hi  - new_lo
        return 100.0 * new_range / orig_range if orig_range > 0 else 0.0

    _gs_pct_lr = _gs_pct_of_original(*_gs_narrowed_lr, *GRID_ORIG_BOUNDS['learning_rate'], is_log=True)
    _gs_pct_wd = _gs_pct_of_original(*_gs_narrowed_wd, *GRID_ORIG_BOUNDS['weight_decay'],  is_log=True)
    _gs_pct_do = _gs_pct_of_original(*_gs_narrowed_do, *GRID_ORIG_BOUNDS['dropout'],       is_log=False)

    print(f"\n  Narrowed bounds (with {GRID_MARGIN_PCT}% margin, clamped to original space):")
    print(f"  {'Dimension':<16} {'Narrowed Range':<36} {'Original Range':<36} {'% of Original':<14}")
    print(f"  {'-'*16} {'-'*36} {'-'*36} {'-'*14}")
    print(f"  {'learning_rate':<16} [{_gs_narrowed_lr[0]:.6e}, {_gs_narrowed_lr[1]:.6e}]  (log)   "
          f"[{GRID_ORIG_BOUNDS['learning_rate'][0]:.6e}, {GRID_ORIG_BOUNDS['learning_rate'][1]:.6e}]  (log)   "
          f"{_gs_pct_lr:>6.1f}%")
    print(f"  {'weight_decay':<16} [{_gs_narrowed_wd[0]:.6e}, {_gs_narrowed_wd[1]:.6e}]  (log)   "
          f"[{GRID_ORIG_BOUNDS['weight_decay'][0]:.6e}, {GRID_ORIG_BOUNDS['weight_decay'][1]:.6e}]  (log)   "
          f"{_gs_pct_wd:>6.1f}%")
    print(f"  {'dropout':<16} [{_gs_narrowed_do[0]:.6e}, {_gs_narrowed_do[1]:.6e}]  (lin)   "
          f"[{GRID_ORIG_BOUNDS['dropout'][0]:.6e}, {GRID_ORIG_BOUNDS['dropout'][1]:.6e}]  (lin)   "
          f"{_gs_pct_do:>6.1f}%")

    # OBS: Denne volume-beregning er kun et informativt print og påvirker ikke algoritmen.
    # Hvis man vil have "andel af volumen" korrekt for 3 dimensioner, vil man typisk dividere med 100^3.
    _gs_total_vol_pct = _gs_pct_lr * _gs_pct_wd * _gs_pct_do / (100 * 100)
    print(f"\n  Combined volume: {_gs_total_vol_pct:.1f}% of original search space")

    # --------------------------------------------------
    # --- Store narrowed bounds + desired grid resolution ---
    # --------------------------------------------------

    # Vi gemmer bounds og opløsning samlet, så grid-cellen kan bygge et narrowed grid direkte.
    narrowed_grid_space = {
        'lr': _gs_narrowed_lr,
        'wd': _gs_narrowed_wd,
        'do': _gs_narrowed_do,
        'n_lr': GRID_N_LR,
        'n_wd': GRID_N_WD,
        'n_do': GRID_N_DO,
    }

    # --------------------------------------------------
    # --- Preview: hvilke punkter ville et narrowed grid bestå af? ---
    # --------------------------------------------------

    _preview_total = GRID_N_LR * GRID_N_WD * GRID_N_DO

    # Log-spaced preview for LR/WD og linear for dropout
    _preview_lr = np.logspace(np.log10(_gs_narrowed_lr[0]),
                              np.log10(_gs_narrowed_lr[1]), GRID_N_LR).tolist()
    _preview_wd = np.logspace(np.log10(_gs_narrowed_wd[0]),
                              np.log10(_gs_narrowed_wd[1]), GRID_N_WD).tolist()
    _preview_do = np.linspace(_gs_narrowed_do[0], _gs_narrowed_do[1], GRID_N_DO).tolist()

    print(f"\n  Narrowed grid preview ({GRID_N_LR}×{GRID_N_WD}×{GRID_N_DO} = {_preview_total} trials):")
    print(f"    LR grid  (log): {[f'{v:.6e}' for v in _preview_lr]}")
    print(f"    WD grid  (log): {[f'{v:.6e}' for v in _preview_wd]}")
    print(f"    Dropout  (lin): {[f'{v:.4f}' for v in _preview_do]}")

    print(f"\n  To use in the grid search cell, set GRID_USE_NARROWED_SPACE = True")

    # --------------------------------------------------
    # --- Cleanup: slet midlertidige variabler fra notebook namespace ---
    # --------------------------------------------------

    del _gs_data, _gs_results, _gs_subset, _gs_ranked, _gs_top
    del _gs_top_lrs, _gs_top_wds, _gs_top_dos, _gs_raw_bounds
    del _preview_lr, _preview_wd, _preview_do

else:
    # Hvis checkpoint-fil ikke findes, så nulstil output-variablen
    if _gs_ckpt_file:
        print(f"ERROR: Checkpoint file {_gs_ckpt_file} not found.")
    narrowed_grid_space = None

### 3D visualisering af grid search checkpoint

I denne celle visualiserer vi resultaterne fra grid search i et interaktivt 3D scatter plot. Vi loader den nyeste grid search checkpoint-fil (JSON) og udtrækker hyperparametrene (learning rate, weight decay, dropout) samt mean CV validation loss.

Learning rate og weight decay plottes på log10-skala for at gøre mønstre tydelige, da de varierer over størrelsesordener. Loss plottes på z-aksen. Farven på hvert punkt repræsenterer loss, og dropout kodes som markørstørrelse, så man kan se om dropout tenderer til at være høj/lav i de bedste områder.

Vi markerer de bedste TOP_X trials (lavest loss) særskilt i rød, så man visuelt kan se hvor de bedste grid-resultater ligger. Hvis vi har beregnet et indsnævret grid-rum i analyse-cellen (6b), tegner vi også en boks, der viser de indsnævrede bounds, så vi kan vurdere om “zoom-in” faktisk dækker området med de bedste punkter.

**Hvad kunne vi ellers have gjort?**
Dropout har ofte få diskrete værdier i grid search, så vi kunne også lave separate plots pr. dropout-værdi for at gøre sammenligningen mere overskuelig. Alternativt kunne vi lave 2D plots (LR vs loss og WD vs loss), som er lettere at forklare mundtligt end et 3D plot.

In [ ]:
# ==========================================
# 6c. 3D Visualisation of Grid Search Checkpoint
# ==========================================
# Formål:
# - Læs grid search checkpoint (JSON)
# - Plot LR vs WD vs Loss i et interaktivt 3D scatter plot
# - Farve: loss (lavere = bedre)
# - Markørstørrelse: dropout (højere dropout = større markør)
# - Highlight top-X trials (fra analyse-cellen 6b) i rød
# - (Valgfrit) vis en bounding box for narrowed_grid_space hvis den findes

import plotly.graph_objects as go
import json as _json_6c

# --------------------------------------------------
# --- Load grid search checkpoint ---
# --------------------------------------------------

# Find nyeste checkpoint-id for grid search
_gv_ckpt_id = _get_grid_checkpoint_id('grid_search', find_latest=True)

# Hvis der ikke findes noget checkpoint, kan vi ikke plotte noget
if _gv_ckpt_id is None:
    raise RuntimeError("No grid search checkpoint found. Run the grid search cell first.")

# Byg filepath til checkpoint
_gv_ckpt_file = os.path.join(DRIVE_DIR, f"grid_search_{_gv_ckpt_id}.json")

# Load JSON checkpoint
with open(_gv_ckpt_file, "r") as f:
    _gv_data = _json_6c.load(f)

# "results" er en liste af dicts (hver dict indeholder params og mean_val_loss)
_gv_results = _gv_data.get("results", [])

# --------------------------------------------------
# --- Restrict to analysed subset (valgfrit) ---
# --------------------------------------------------

# Hvis GRID_ANALYSE_FIRST_N_TRIALS findes (fra 6b), plotter vi kun de første N trials
# Ellers plotter vi alle results i checkpoint
_gv_n = min(GRID_ANALYSE_FIRST_N_TRIALS, len(_gv_results)) if 'GRID_ANALYSE_FIRST_N_TRIALS' in dir() else len(_gv_results)
_gv_results = _gv_results[:_gv_n]

# --------------------------------------------------
# --- Extract arrays for plotting ---
# --------------------------------------------------

# Udpak LR/WD/Dropout/Loss til numpy-arrays
_gv_lr   = np.array([r["params"][0] for r in _gv_results])
_gv_wd   = np.array([r["params"][1] for r in _gv_results])
_gv_do   = np.array([r["params"][2] for r in _gv_results])
_gv_loss = np.array([r["mean_val_loss"] for r in _gv_results])

# --------------------------------------------------
# --- Scale dropout to marker size ---
# --------------------------------------------------

# Vi skalerer dropout til markørstørrelser fra 4 til 18
# (hvis dropout er konstant, bruger vi en fast størrelse)
_gv_do_min, _gv_do_max = _gv_do.min(), _gv_do.max()

if _gv_do_max > _gv_do_min:
    _gv_sizes = 4 + 14 * (_gv_do - _gv_do_min) / (_gv_do_max - _gv_do_min)
else:
    _gv_sizes = np.full_like(_gv_do, 10.0)

# --------------------------------------------------
# --- Identify top-X points (lavest loss) ---
# --------------------------------------------------

# Antal top-punkter kommer fra GRID_TOP_X hvis den findes (fra 6b), ellers default 5
_gv_top_count = GRID_TOP_X if 'GRID_TOP_X' in dir() else 5

# Sortér indices efter loss (lavest først) og tag top-X
_gv_sorted_idx = np.argsort(_gv_loss)
_gv_top_idx = set(_gv_sorted_idx[:_gv_top_count])

# Boolean maske: True for top-X punkter
_gv_is_top = np.array([i in _gv_top_idx for i in range(len(_gv_loss))])

# --------------------------------------------------
# --- Hover text (interaktiv info) ---
# --------------------------------------------------

# Plotly hover-tekst for hver trial (viser trialnr, parametre og loss)
_gv_hover = [
    f"Trial {_gv_results[i].get('trial', i+1)}<br>LR: {_gv_lr[i]:.6e}<br>"
    f"WD: {_gv_wd[i]:.6e}<br>Dropout: {_gv_do[i]:.4f}<br>Loss: {_gv_loss[i]:.4f}"
    for i in range(len(_gv_loss))
]

# --------------------------------------------------
# --- Build figure ---
# --------------------------------------------------

fig = go.Figure()

# 1) Alle "ikke-top" trials (farves efter loss)
# Vi plotter LR og WD på log10-skala for at se mønstre over størrelsesordener
fig.add_trace(go.Scatter3d(
    x=np.log10(_gv_lr[~_gv_is_top]),
    y=np.log10(_gv_wd[~_gv_is_top]),
    z=_gv_loss[~_gv_is_top],
    mode='markers',
    marker=dict(
        size=_gv_sizes[~_gv_is_top],
        color=_gv_loss[~_gv_is_top],
        colorscale='Viridis',
        colorbar=dict(title='Loss', x=1.05),
        opacity=0.6,
        line=dict(width=0.5, color='white'),
    ),
    # Hovertekst for ikke-top punkter
    text=[h for h, t in zip(_gv_hover, _gv_is_top) if not t],
    hoverinfo='text',
    name='Grid Trials',
))

# 2) Top-X trials (highlightet i rød)
fig.add_trace(go.Scatter3d(
    x=np.log10(_gv_lr[_gv_is_top]),
    y=np.log10(_gv_wd[_gv_is_top]),
    z=_gv_loss[_gv_is_top],
    mode='markers',
    marker=dict(
        size=_gv_sizes[_gv_is_top] + 4,
        color='red',
        opacity=0.9,
        symbol='diamond',
        line=dict(width=1, color='darkred'),
    ),
    # Hovertekst for top punkter
    text=[h for h, t in zip(_gv_hover, _gv_is_top) if t],
    hoverinfo='text',
    name=f'Top {_gv_top_count}',
))

# --------------------------------------------------
# --- Draw bounding box for narrowed grid (valgfrit) ---
# --------------------------------------------------

# Hvis narrowed_grid_space findes (fra 6b), tegner vi en boks i LR/WD (log10) og i loss-aksen
if 'narrowed_grid_space' in dir() and narrowed_grid_space is not None:

    # Narrowed bounds for LR/WD (log10)
    _gv_nb_lr = [np.log10(narrowed_grid_space['lr'][0]), np.log10(narrowed_grid_space['lr'][1])]
    _gv_nb_wd = [np.log10(narrowed_grid_space['wd'][0]), np.log10(narrowed_grid_space['wd'][1])]

    # Z (loss) bounds vælges som [min-0.01, max+0.01] så boksen spænder hele loss-rangen
    _gv_nb_z_lo = float(_gv_loss.min()) - 0.01
    _gv_nb_z_hi = float(_gv_loss.max()) + 0.01

    # Hjælpefunktion: returnér de 12 kanter for en 3D-boks (Plotly tegner linjer)
    def _gv_box_edges(x0, x1, y0, y1, z0, z1):
        edges_x, edges_y, edges_z = [], [], []
        for (xa, ya, za), (xb, yb, zb) in [
            ((x0,y0,z0),(x1,y0,z0)), ((x0,y1,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x1,y0,z1)), ((x0,y1,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y1,z0)), ((x1,y0,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x0,y1,z1)), ((x1,y0,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y0,z1)), ((x1,y0,z0),(x1,y0,z1)),
            ((x0,y1,z0),(x0,y1,z1)), ((x1,y1,z0),(x1,y1,z1)),
        ]:
            edges_x += [xa, xb, None]
            edges_y += [ya, yb, None]
            edges_z += [za, zb, None]
        return edges_x, edges_y, edges_z

    bx, by, bz = _gv_box_edges(
        _gv_nb_lr[0], _gv_nb_lr[1],
        _gv_nb_wd[0], _gv_nb_wd[1],
        _gv_nb_z_lo, _gv_nb_z_hi
    )

    fig.add_trace(go.Scatter3d(
        x=bx, y=by, z=bz,
        mode='lines',
        line=dict(color='orange', width=3),
        name='Narrowed bounds',
        hoverinfo='skip',
    ))

# --------------------------------------------------
# --- Layout / labels ---
# --------------------------------------------------

fig.update_layout(
    title=f'Grid Search Trials (first {_gv_n}) — Marker size ∝ Dropout',
    scene=dict(
        xaxis_title='log₁₀(Learning Rate)',
        yaxis_title='log₁₀(Weight Decay)',
        zaxis_title='Mean CV Val Loss',
    ),
    width=900,
    height=700,
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

# --------------------------------------------------
# --- Cleanup ---
# --------------------------------------------------

del _gv_data, _gv_results, _gv_lr, _gv_wd, _gv_do, _gv_loss
del _gv_sizes, _gv_sorted_idx, _gv_top_idx, _gv_is_top, _gv_hover